In [1]:
import os
import sys
import re
import ast
import json
from typing import Dict, Any, List, Optional
from dataclasses import dataclass

import pandas as pd
from pprint import pprint

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(project_root)

from src.dao import YouTubeDBSetup

In [2]:
df = pd.read_csv("./20250902_213040/batch_1_results.csv")
print(df.columns)

Index(['comment_id', 'is_hate_speech', 'categories', 'similar_cases_used',
       'target_group', 'hate_type', 'used_prompt', 'classification_result'],
      dtype='object')


In [3]:
@dataclass
class HateSpeechResult:
    """혐오표현 분류 결과를 담는 데이터클래스"""
    input_text: str
    is_hate_speech: bool
    evidence_strength: float
    reasoning: str
    categories: List[str]
    similar_cases_used: List[str]
    prompt: Optional[str] = None
    target_group: Optional[str] = None
    hate_type: Optional[str] = None

    def to_dict(self) -> Dict[str, Any]:
        """데이터클래스를 딕셔너리로 변환"""
        return {
            'prompt': self.prompt,
            'input_text': self.input_text,
            'is_hate_speech': self.is_hate_speech,
            'categories': self.categories,
            'evidence_strength': self.evidence_strength,
            'reasoning': self.reasoning,
            'similar_cases_used': self.similar_cases_used,
            'target_group': self.target_group,
            'hate_type': self.hate_type
        }

    def to_json(self, indent: int = 2) -> str:
        """JSON 문자열로 변환"""
        return json.dumps(self.to_dict(), ensure_ascii=False, indent=indent)

def parse_with_regex(text: str) -> Dict[str, Any]:
    """정규표현식을 사용하여 키=값 패턴을 파싱"""
    result = {}
    
    # prompt 추출 (특별 처리 - 매우 긴 문자열)
    prompt_pattern = r"prompt='(.*?)' input_text="
    prompt_match = re.search(prompt_pattern, text, re.DOTALL)
    if prompt_match:
        result['prompt'] = prompt_match.group(1)
    else:
        result['prompt'] = None
    
    # 나머지 필드들을 정규표현식으로 추출
    patterns = {
        'input_text': r"input_text='([^']*)'",
        'is_hate_speech': r"is_hate_speech=(\w+)",
        'categories': r"categories=(\[.*?\])",
        'evidence_strength': r"evidence_strength=([\d.]+)",
        'reasoning': r"reasoning='([^']*)'",
        'similar_cases_used': r"similar_cases_used=(\[.*?\])",
        'target_group': r"target_group='([^']*)'",
        'hate_type': r"hate_type='([^']*)'"
    }
    
    for key, pattern in patterns.items():
        match = re.search(pattern, text, re.DOTALL)
        if match:
            value = match.group(1)
            
            # 데이터 타입 변환
            if key == 'is_hate_speech':
                result[key] = value == 'True'
            elif key in ['categories', 'similar_cases_used']:
                try:
                    result[key] = ast.literal_eval(value)
                except:
                    result[key] = []
            elif key == 'evidence_strength':
                result[key] = float(value)
            else:
                result[key] = value
        else:
            # 기본값 설정
            if key in ['categories', 'similar_cases_used']:
                result[key] = []
            elif key == 'is_hate_speech':
                result[key] = False
            elif key == 'evidence_strength':
                result[key] = 0.0
            else:
                result[key] = None
    
    return result

class ResultParser:
    """결과 파서 클래스"""
    
    @staticmethod
    def parse(text: str) -> Dict[str, Any]:
        """텍스트를 파싱하여 딕셔너리로 반환"""
        return parse_with_regex(text)
    
    @staticmethod
    def parse_to_dataclass(text: str) -> HateSpeechResult:
        """텍스트를 파싱하여 데이터클래스로 반환"""
        parsed_dict = parse_with_regex(text)
        return HateSpeechResult(**parsed_dict)
    
    @staticmethod
    def parse_to_json_string(text: str) -> str:
        """텍스트를 파싱하여 JSON 문자열로 반환"""
        result = ResultParser.parse(text)
        return json.dumps(result, ensure_ascii=False, indent=2)

In [4]:
result = ResultParser.parse(df.loc[0, 'classification_result'])
print(result['input_text'])
print(result['is_hate_speech'])
print(result['reasoning'])
print(result['categories'])


"여러분 토욜 서초역 7번출구 2시에 집회 합니다 윤카 사저앞까지 행진도 합니다 많은분들 참석 바랍니다"
False
집회 참여를 독려하는 내용으로, 특정 보호 특성(성별, 연령, 인종, 출신지역, 성적지향, 종교 등)을 이유로 한 모욕·비하·멸시·위협 또는 차별/폭력 선동이 없다. 욕설도 포함되어 있지 않다.
['혐오없음']


In [5]:
dao = YouTubeDBSetup()
dao.update_hate_speech_analysis_by_comment(result['input_text'], result)


⚠️ 해당 텍스트를 찾을 수 없습니다: "여러분 토욜 서초역 7번출구 2시에 집회 합니다 윤카 사저앞까지 행진도 합니다 많은분들 참석 바랍니다"


False